# External benchmark: MAVOS-DD test sample, by generator, language and compression tier

Scores the released system on a stratified sample of the MAVOS-DD test split (7+ generators, 8 languages, YouTube reals), re-encoded at H.264 CRF 23 and 40 as extra tiers, and reports by generator, language and source video with video-clustered intervals. Accept the dataset terms at https://huggingface.co/datasets/unibuc-cs/MAVOS-DD with your account first; the login cell asks for a token.

In [ ]:
!nvidia-smi -L; nproc
%cd /content
!rm -rf repo && git clone -q -b revision/round-3 https://github.com/saoirsebarry/multiagent-deepfake-detection.git repo
%cd /content/repo
!pip -q install speechbrain timm librosa opencv-python-headless mtcnn datasets huggingface_hub
!apt-get -qq install -y ffmpeg > /dev/null
!git log --oneline -1
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
from huggingface_hub import login
login()  # paste a read token from https://huggingface.co/settings/tokens

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/polyglotfake/external/mavos'
os.makedirs(OUT, exist_ok=True)

Only the metadata files are snapshotted (a few MB); the selected videos are downloaded one by one into Drive, so a recycled runtime resumes where it stopped instead of starting over. The download needs no GPU; switch the runtime to a T4 before the staging cell.


In [ ]:
from huggingface_hub import snapshot_download
SNAP = snapshot_download('unibuc-cs/MAVOS-DD', repo_type='dataset', local_dir='/content/mavos_meta', allow_patterns=['*.json', '*.arrow', '*.py', 'README.md'])
!python -u tools/external_benchmark/mavos_manifest.py --snapshot /content/mavos_meta --out "$OUT/videos.csv" --cap 40 --real_cap 40 --video_dir "$OUT/videos"


## Stage clips (original, CRF 23, CRF 40)

In [ ]:
import os
N = max(2, os.cpu_count() // 2)
os.environ["N"] = str(N)
print("staging shards:", N)
!seq 0 $((N-1)) | xargs -P $N -I{} sh -c 'python -u tools/external_benchmark/stage_videos.py --manifest "$OUT/videos.csv" --out_dir /content/mavos_clips --crf 23 40 --shard {}/$N > /content/stage_{}.log 2>&1'
!tail -n 3 /content/stage_*.log
!ls /content/mavos_clips/test | wc -l; ls /content/mavos_clips/crf23/test | wc -l; ls /content/mavos_clips/crf40/test | wc -l
!cp /content/mavos_clips/metadata.csv "$OUT/"

## Score with the released system and report

In [ ]:
!mkdir -p "$OUT/original" "$OUT/crf23" "$OUT/crf40"
!python -u tools/source_disjoint/score_split.py --data_dir /content/mavos_clips --split test --ckpt_dir checkpoints --out "$OUT/original/scores.csv" 2>&1 | tail -n 5
!python -u tools/source_disjoint/score_split.py --data_dir /content/mavos_clips/crf23 --split test --ckpt_dir checkpoints --out "$OUT/crf23/scores.csv" 2>&1 | tail -n 5
!python -u tools/source_disjoint/score_split.py --data_dir /content/mavos_clips/crf40 --split test --ckpt_dir checkpoints --out "$OUT/crf40/scores.csv" 2>&1 | tail -n 5


In [ ]:
!python -u tools/external_benchmark/report_by_group.py --scores "$OUT/original/scores.csv" "$OUT/crf23/scores.csv" "$OUT/crf40/scores.csv" --metadata "$OUT/metadata.csv" --group_by generator language source_video --cluster source_video --tau 0.35 --out "$OUT/report.json"